In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("rawdata-to-star-optimized") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
    .getOrCreate()

pg_url = "jdbc:postgresql://postgres:5432/bigdata_spark"
pg_props = {
    "user": "user", 
    "password": "password", 
    "driver": "org.postgresql.Driver"
}

raw_df = spark.read.jdbc(url=pg_url, table="raw_data", properties=pg_props)
raw_df.createOrReplaceTempView("raw_data")

In [2]:
spark.sql("""
    SELECT 
        customer_first_name as first_name, 
        first(customer_last_name) as last_name,
        first(customer_age) as age, 
        first(customer_pet_type) as pet_type, 
        first(customer_pet_breed) as pet_breed, 
        first(pet_category) as pet_category,
        first(customer_country) as country, 
        CAST(NULL AS STRING) as city
    FROM raw_data
    GROUP BY customer_email, customer_first_name 
""").write.jdbc(url=pg_url, table="dim_customers", mode="append", properties=pg_props)

cust_df = spark.read.jdbc(url=pg_url, table="dim_customers", properties=pg_props)
cust_df.createOrReplaceTempView("dim_customers")

In [3]:
spark.sql("""
    SELECT 
        seller_first_name as first_name, 
        first(seller_last_name) as last_name,
        first(seller_country) as country
    FROM raw_data
    GROUP BY seller_email, seller_first_name
""").write.jdbc(url=pg_url, table="dim_sellers", mode="append", properties=pg_props)

sellers_df = spark.read.jdbc(url=pg_url, table="dim_sellers", properties=pg_props)
sellers_df.createOrReplaceTempView("dim_sellers")

In [4]:
spark.sql("""
    SELECT 
        store_name as name, 
        first(store_country) as country, 
        first(store_city) as city
    FROM raw_data
    GROUP BY store_name
""").write.jdbc(url=pg_url, table="dim_stores", mode="append", properties=pg_props)

stores_df = spark.read.jdbc(url=pg_url, table="dim_stores", properties=pg_props)
stores_df.createOrReplaceTempView("dim_stores")

In [5]:
spark.sql("""
    SELECT 
        product_name as name, 
        first(product_category) as category, 
        first(product_brand) as brand, 
        first(product_price) as price, 
        first(product_rating) as rating,
        first(supplier_name) as supplier_name,
        first(supplier_country) as supplier_country
    FROM raw_data
    GROUP BY product_name
""").write.jdbc(url=pg_url, table="dim_products", mode="append", properties=pg_props)

prod_df = spark.read.jdbc(url=pg_url, table="dim_products", properties=pg_props)
prod_df.createOrReplaceTempView("dim_products")

In [6]:
spark.sql("""
    SELECT 
        r.sale_date, 
        c.customer_id, 
        sel.seller_id, 
        p.product_id, 
        st.store_id, 
        r.sale_quantity as quantity, 
        r.sale_total_price as total_price
    FROM raw_data r
    JOIN dim_customers c ON r.customer_first_name = c.first_name 
         AND r.customer_last_name = c.last_name
    JOIN dim_sellers sel ON r.seller_first_name = sel.first_name 
         AND r.seller_country = sel.country
    JOIN dim_products p ON r.product_name = p.name
    JOIN dim_stores st ON r.store_name = st.name
""").write.jdbc(url=pg_url, table="fact_sales", mode="append", properties=pg_props)